In [16]:
import os
import numpy as np
from datetime import datetime, timedelta
import uuid
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import dotenv
dotenv.load_dotenv()

True

In [19]:
class Baseline:
    def __init__(self):
        qdrant_url = os.getenv("QDRANT_URL")
        qdrant_api_key = os.getenv("QDRANT_API_KEY")

        if not qdrant_url or not qdrant_api_key:
            raise RuntimeError(
                "QDRANT_URL and QDRANT_API_KEY must be set as environment variables"
            )

        self.client = QdrantClient(
            url=qdrant_url,
            api_key=qdrant_api_key,
            timeout=60,
            check_compatibility=False
        )

        self.base_timestamp = datetime.now() - timedelta(days=180)

    def setup_collections(self):
        print("Setting up Qdrant Cloud collections...")

        collections = {
            "sensor_baseline": VectorParams(size=128, distance=Distance.COSINE),
            "visual_baseline": VectorParams(size=512, distance=Distance.COSINE),
            "audio_baseline": VectorParams(size=512, distance=Distance.COSINE),
        }

        for name, config in collections.items():
            try:
                self.client.create_collection(
                    collection_name=name,
                    vectors_config=config,
                )
                print(f"Created collection: {name}")
            except Exception:
                print(f"Collection already exists or cannot be created: {name}")

    def generate_sensor_baseline(self, n_samples=2000):
        points = []
    
        temp_state = np.array([20.0, 20.2, 19.8])   # AR(1) state
        pressure_state = np.array([1007.5, 1007.6, 1007.4])

        for i in range(n_samples):
            timestamp = self.base_timestamp + timedelta(hours=i * 0.5)
            corner_id = np.random.choice(["corner_1", "corner_2", "corner_3"])
            shift = ["morning", "afternoon", "night"][i % 3]

            diurnal = 2.0 * np.sin(2 * np.pi * (i % 48) / 48)

            temp_state = 0.95 * temp_state + np.random.normal(0, 0.2, 3)
            pressure_state = 0.98 * pressure_state + np.random.normal(0, 0.5, 3)

            temperature = np.clip(temp_state + diurnal, 18, 26)
            pressure = np.clip(pressure_state, 995, 1020)

            embedding = self._create_sensor_embedding(temperature, pressure, shift)

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding.tolist(),
                payload={
                    "timestamp": timestamp.isoformat(),
                    "corner_id": corner_id,
                    "shift": shift,
                    "avg_temperature": float(np.mean(temperature)),
                    "avg_pressure": float(np.mean(pressure)),
                    "condition": "normal"
                }
            ))

        self.client.upsert("sensor_baseline", points)


    def _create_sensor_embedding(self, temperature, pressure, shift):
        raw = np.concatenate([temperature, pressure])

        stats = np.array([
            np.mean(temperature),
            np.std(temperature),
            np.min(temperature),
            np.max(temperature),
            np.mean(pressure),
            np.std(pressure),
            np.min(pressure),
            np.max(pressure),
            1.0 if shift == "morning" else 0.0,
            1.0 if shift == "afternoon" else 0.0,
            1.0 if shift == "night" else 0.0,
        ])

        rolling = np.random.normal(0, 0.1, 128 - len(raw) - len(stats))
        emb = np.concatenate([raw, stats, rolling])
        return emb / (np.linalg.norm(emb) + 1e-8)

    def generate_visual_baseline(self, n_samples=1500):
        points = []
    
        scene_probs = {
            "worker_monitoring_panel": 0.4,
            "worker_walking_inspection": 0.2,
            "machinery_idle_running": 0.15,
            "clean_work_area": 0.1,
            "routine_maintenance": 0.05,
            "equipment_monitoring": 0.05,
            "safety_check": 0.05,
        }
        
        scenes = list(scene_probs.keys())
        probs = list(scene_probs.values())

        for i in range(n_samples):
            timestamp = self.base_timestamp + timedelta(minutes=i * 5)
            scene_type = np.random.choice(scenes, p=probs)

            embedding = self._create_visual_embedding(scene_type)

            lighting = np.random.choice(["bright", "normal", "dim"], p=[0.2, 0.6, 0.2])

            ppe_ok = np.random.rand() > 0.02  # 98% compliance

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding.tolist(),
                payload={
                    "timestamp": timestamp.isoformat(),
                    "scene_type": scene_type,
                    "ppe_compliance": ppe_ok,
                    "lighting": lighting,
                    "camera_noise_level": float(np.random.uniform(0, 0.2)),
                    "condition": "normal"
                }
            ))

        self.client.upsert("visual_baseline", points)

    def _create_visual_embedding(self, scene_type):
        base = np.random.randn(512) * 0.3
        emb = base + np.random.randn(512) * 0.1
        return emb / (np.linalg.norm(emb) + 1e-8)

    def generate_audio_baseline(self, n_samples=1200):
        points = []

        base_noise = 50  

        for i in range(n_samples):
            timestamp = self.base_timestamp + timedelta(minutes=i * 7)

            sound_types = np.random.choice(
                ["machinery", "ventilation", "speech", "footsteps"],
                size=np.random.randint(1, 3),
                replace=False
            )

            embedding = np.mean([self._create_audio_embedding(s) for s in sound_types], axis=0)

            volume_db = base_noise + np.random.normal(0, 5)

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding.tolist(),
                payload={
                    "timestamp": timestamp.isoformat(),
                    "sound_mix": sound_types.tolist(),
                    "volume_db": float(volume_db),
                    "alarm_detected": False,
                    "condition": "normal"
                }
            ))

        self.client.upsert("audio_baseline", points)


    def _create_audio_embedding(self, sound_type):
        base = np.random.randn(512) * 0.4
        emb = base + np.random.randn(512) * 0.15
        return emb / (np.linalg.norm(emb) + 1e-8)

In [20]:
def main():
    generator = Baseline()

    generator.setup_collections()
    generator.generate_sensor_baseline(2000)
    generator.generate_visual_baseline(1500)
    generator.generate_audio_baseline(1200)

    print("Baseline generation complete (Qdrant Cloud)")


if __name__ == "__main__":
    main()


Setting up Qdrant Cloud collections...
Collection already exists or cannot be created: sensor_baseline
Collection already exists or cannot be created: visual_baseline
Collection already exists or cannot be created: audio_baseline


ResponseHandlingException: Server disconnected without sending a response.